In [8]:
from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel
from langchain.agents import create_agent

import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()
# model setting
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
MODEL_NAME = "gpt-4.1-mini"

In [9]:
from typing import Literal
from pydantic import BaseModel, Field


class Vector3Model(BaseModel):
    """3D world position."""

    x: float = Field(..., description="World x coordinate.")
    y: float = Field(..., description="World y coordinate.")
    z: float = Field(..., description="World z coordinate.")


class CommandActionModel(BaseModel):
    """One executable action for a Unity NPC command."""

    action_id: str | None = Field(
        ...,
        description="Stable action id. Use null when the backend should assign one.",
    )
    command: Literal["MOVE_TO", "GET_ITEM", "PUT_ITEM", "STOP"] | None = Field(
        ...,
        description="Executable Unity command. Use MOVE_TO, GET_ITEM, PUT_ITEM, STOP, or null.",
    )
    object_name: str | None = Field(
        ...,
        description="Target object or place name in English. Use null if the action has no named target.",
    )
    object_id: str | None = Field(
        ...,
        description="Resolved Unity object id for this action. Use null until Unity resolves a concrete object id.",
    )
    position: Vector3Model | None = Field(
        ...,
        description="Target world position for coordinate movement. Use null when moving to a named object.",
    )


class CommandModel(BaseModel):
    """User natural language command converted into a command for one NPC."""

    actions: list[CommandActionModel] = Field(
        ...,
        description="Ordered executable Unity actions. Use an empty list for non-executable input.",
    )
    message: str = Field(..., description="AI response message for the user.")

In [11]:
agent = create_agent(
    model=MODEL_NAME,
    tools=[],
    system_prompt=(
        "You are a command planner for a Unity NPC. "
        "Return only executable Unity commands. "
        "The message must be Korean."
    ),
    response_format=CommandModel,
)